# OSCAR Baseline Notebook — BLIP/CLIP embeddings followed by SD generation

This notebook is a scaled implementation for our baseline, the OSCAR model. The current implementation is designed for **domain net**.

Pipeline: **real images → BLIP captions → CLIP text embeddings → Stable Diffusion synthesis**

The notebook uses a hyperparameter **topk** to decide how many embeddings are used to create the synthetic data. Notebook outputs are used by part B to evaluate accuracies. 

## 1) Setup

If a Kaggle image already has the needed packages, you can leave the install line commented.

In [ ]:
!pip -q install safetensors
!pip install -q "transformers==4.45.2" "accelerate==0.34.2"
!pip install -q "diffusers==0.29.2"
!pip install torch torchvision torchaudio

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")


import os
import json
import math
import random
import shutil
import warnings
from collections import defaultdict, Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import datetime

import transformers as _tr, diffusers as _df, logging as _pylog
_tr.logging.set_verbosity_error()
_df.logging.set_verbosity_error()
_pylog.getLogger("diffusers").setLevel(_pylog.ERROR)
_pylog.getLogger("transformers").setLevel(_pylog.ERROR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 705.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.1 MB/s eta 0:00:00


In [ ]:

SEED = 28
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

NUM_GPUS = torch.cuda.device_count()
GPU0 = torch.device("cuda:0") if NUM_GPUS >= 1 else torch.device("cpu")
GPU1 = torch.device("cuda:1") if NUM_GPUS >= 2 else GPU0   # falls back to GPU0 if only 1

print("Device:", DEVICE)
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {props.name}  {props.total_memory // 1024**2} MB")
if NUM_GPUS < 2:
    print("  ⚠️  Only 1 GPU found — SD will run single-GPU (no parallelism)")
print("Torch:", torch.__version__)


Device: cuda
GPUs available: 2
  cuda:0  Tesla T4  14912 MB
  cuda:1  Tesla T4  14912 MB
Torch: 2.10.0+cu128


## 2) Configuration

This notebook uses **NICO++ only** and selects the **top 30 classes by image count**.

### Key changes
- `weight_decay`, `dropout_p`, `patience`, `grad_clip` added for regularization and early stopping.
- `mix_real_with_synthetic`: when True, real training images are combined with synthetic ones.

In [ ]:
@dataclass
class Config:
    dataset_name: str = "domainnet"
    data_root: str = "/kaggle/input/datasets/mei1963/domainnet/DomainNet" #replace path to the databset if required


    max_classes: int = 12
    images_per_class_train: int = 400
    images_per_class_test: int = 100
    captions_per_class: int = 20        

    num_clients: int = 6

    synth_images_per_class: int = 200    
    diffusion_steps: int = 45             
    guidance_scale: float = 5.0           

    # Top-K compression for CLIP embeddings
    topk_k: int = 64

    batch_size: int = 32
    epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-2
    dropout_p: float = 0.2     
    grad_clip: float = 1.0
    patience: int = 10

    blip_model: str = "Salesforce/blip-image-captioning-base"
    clip_model: str = "openai/clip-vit-base-patch32"
    sd_model: str = "runwayml/stable-diffusion-v1-5"
    backbone: str = "resnet18"

    work_dir: str = "/kaggle/working/oscar_baseline_domainnet"
    outputs_dir: str = "/kaggle/working/outputs"

CFG = Config()

WORK = Path(CFG.work_dir)
WORK.mkdir(parents=True, exist_ok=True)
for sub in ["captions", "embeddings", "synthetic", "checkpoints", "metrics"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

OUTPUTS = Path(CFG.outputs_dir)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print("Config ready. Work dir:", WORK)
print("Outputs dir:", OUTPUTS)
print(f"Estimated SD images: {CFG.max_classes} classes x {CFG.synth_images_per_class} imgs x {CFG.diffusion_steps} steps = {CFG.max_classes * CFG.synth_images_per_class * CFG.diffusion_steps:,} total steps")


Config ready. Work dir: /kaggle/working/oscar_baseline_domainnet
Outputs dir: /kaggle/working/outputs
Estimated SD images: 12 classes x 200 imgs x 45 steps = 108,000 total steps


## 3) Dataset utilities


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

def sample_list(items, k, rng):
    items = list(items)
    if k is None or k >= len(items):
        return items
    return rng.sample(items, k)

def count_class_images(root: Path):
    counts = []
    for d in sorted([x for x in root.iterdir() if x.is_dir()]):
        n = sum(1 for p in d.rglob("*") if p.is_file() and is_image_file(p))
        if n > 0:
            counts.append((d.name, n))
    counts.sort(key=lambda x: (-x[1], x[0]))
    return counts
def load_split_from_txt(root: Path, split: str):
    records = []
    for txt in sorted(root.glob(f"*_{split}.txt")):
        with open(txt) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rel_path = line.split()[0]  # e.g. clipart/airplane/img001.jpg
                parts = Path(rel_path).parts 
                if len(parts) < 3:
                    continue
                cls = parts[1]
                abs_path = root / rel_path
                if abs_path.exists():
                    records.append((abs_path, cls, split))
    return records

def select_top_classes(root: Path, max_classes: int):
    """
   Counts images per class across all domains,
    returns top max_classes by image count.
    """
    train_records = load_split_from_txt(root, "train")
    counts = Counter(cls for _, cls, _ in train_records)
    top = [cls for cls, _ in counts.most_common(max_classes)]
    class_counts = [(cls, counts[cls]) for cls in top]
    return top, class_counts

def build_imagefolder_records(root: Path, selected_classes, images_per_class_train: int, images_per_class_test: int, seed: int = 42):
    rng = random.Random(seed)
    cls_set = set(selected_classes)

    raw_train = [(p, cls, "train") for p, cls, _ in load_split_from_txt(root, "train") if cls in cls_set]
    raw_test  = [(p, cls, "test")  for p, cls, _ in load_split_from_txt(root, "test")  if cls in cls_set]

    train_by_cls = defaultdict(list)
    test_by_cls  = defaultdict(list)
    for p, cls, _ in raw_train:
        train_by_cls[cls].append(p)
    for p, cls, _ in raw_test:
        test_by_cls[cls].append(p)

    train_records, test_records = [], []
    for cls in selected_classes:
        tr = sorted(train_by_cls[cls])
        te = sorted(test_by_cls[cls])
        rng.shuffle(tr)
        rng.shuffle(te)
        tr = tr[:images_per_class_train]
        te = te[:images_per_class_test]
        train_records.extend([(p, cls, "train") for p in tr])
        test_records.extend([(p, cls, "test")  for p in te])

    return train_records, test_records

class LabeledImageDataset(Dataset):
    def __init__(self, records, class_to_idx, transform=None):
        self.records = records
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, cls, _split = self.records[idx]
        image = load_image(Path(path))
        if self.transform is not None:
            image = self.transform(image)
        label = self.class_to_idx[cls]
        return image, label

def split_classes_across_clients(classes, num_clients):
    clients = [[] for _ in range(num_clients)]
    for i, cls in enumerate(classes):
        clients[i % num_clients].append(cls)
    return clients

def group_records_by_class(records):
    grouped = defaultdict(list)
    for p, cls, split in records:
        grouped[cls].append((p, split))
    return grouped

root = Path(CFG.data_root)
selected_classes, class_counts = select_top_classes(root, CFG.max_classes)
train_records, test_records = build_imagefolder_records(
    root,
    selected_classes=selected_classes,
    images_per_class_train=CFG.images_per_class_train,
    images_per_class_test=CFG.images_per_class_test,
    seed=SEED,
)

class_to_idx = {cls: i for i, cls in enumerate(selected_classes)}
train_grouped = group_records_by_class(train_records)
test_grouped = group_records_by_class(test_records)

print("Selected classes:", selected_classes)
print("Top class counts:", class_counts[:5])
print("Train records:", len(train_records))
print("Test records:", len(test_records))

Selected classes: ['tree', 'golf_club', 'dog', 'spreadsheet', 'squirrel', 'whale', 'tiger', 'table', 'shoe', 'bird', 'windmill', 'beard']
Top class counts: [('tree', 1926), ('golf_club', 1899), ('dog', 1817), ('spreadsheet', 1771), ('squirrel', 1760)]
Train records: 4800
Test records: 1200


## 4) Client simulation

OSCAR is one-shot federated learning, so we keep a small client split for the report and for later ablations.

In [6]:
client_classes = split_classes_across_clients(selected_classes, CFG.num_clients)
for i, cls_list in enumerate(client_classes):
    print(f"Client {i}: {cls_list}")

Client 0: ['tree', 'tiger']
Client 1: ['golf_club', 'table']
Client 2: ['dog', 'shoe']
Client 3: ['spreadsheet', 'bird']
Client 4: ['squirrel', 'windmill']
Client 5: ['whale', 'beard']


## 5) BLIP captioning

We caption only a small number of images per class.  
That keeps the notebook practical on a T4 while still using the OSCAR-style frozen vision-language step.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(CFG.blip_model)
blip_model = BlipForConditionalGeneration.from_pretrained(
    CFG.blip_model,
    torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
blip_model.eval()

def caption_image(image_path: Path, max_new_tokens: int = 20):
    image = load_image(image_path)
    inputs = blip_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.decode(out[0], skip_special_tokens=True).strip()

captions_by_class = {}
caption_path = WORK / "captions" / f"{CFG.dataset_name}_captions.json"

print(f"Captioning {len(selected_classes)} classes ({CFG.captions_per_class} images each)...")
for i, cls in enumerate(selected_classes):
    cls_paths = [p for p, _split in train_grouped.get(cls, [])]
    cls_paths = sorted(cls_paths)[:CFG.captions_per_class]
    entries = []
    for p in cls_paths:
        try:
            cap = caption_image(p)
        except Exception:
            cap = f"a photo of {cls}"
        entries.append({"image": str(p), "caption": cap})
    captions_by_class[cls] = entries
    if (i + 1) % 5 == 0 or (i + 1) == len(selected_classes):
        print(f"  Captioned {i+1}/{len(selected_classes)} classes")

with open(caption_path, "w") as f:
    json.dump(captions_by_class, f, indent=2)

import shutil as _shutil
_shutil.copy(caption_path, OUTPUTS / caption_path.name)

print(f"BLIP captions saved -> {caption_path}")
print(f"BLIP captions saved -> {OUTPUTS / caption_path.name}")


Loading BLIP...


2026-05-07 08:18:05.607380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778141885.773077      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778141885.821074      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778141886.199908      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778141886.199956      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778141886.199959      23 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Captioning 12 classes (20 images each)...
  Captioned 5/12 classes
  Captioned 10/12 classes
  Captioned 12/12 classes
BLIP captions saved -> /kaggle/working/oscar_baseline_domainnet/captions/domainnet_captions.json
BLIP captions saved -> /kaggle/working/outputs/domainnet_captions.json


## 6) CLIP text embeddings

The captions are mapped into CLIP text space and saved to disk after applying compression.  

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
import torch
import numpy as np
import json
import shutil

print("Loading SD-compatible CLIP model (768d)...")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
text_encoder.eval()

def topk_compress_tensor(tensor, k: int):
    if k is None or k >= tensor.shape[-1]:
        return tensor.clone()
    out = torch.zeros_like(tensor)
    _, idx = torch.topk(tensor.abs(), k, dim=-1)
    out.scatter_(-1, idx, tensor.gather(-1, idx))
    return out

embeddings_by_class = {}
emb_dir = WORK / "embeddings" / CFG.dataset_name
emb_dir.mkdir(parents=True, exist_ok=True)

topk_label = f"topk{CFG.topk_k}" if CFG.topk_k is not None else "full"
print(f"Generating CLIP embeddings for {len(selected_classes)} classes (compression: {topk_label})...")

for i, cls in enumerate(selected_classes):
    texts = [x["caption"] for x in captions_by_class.get(cls, []) if x["caption"]]
    if not texts:
        texts = [f"a photo of {cls}"]
    per_caption_embs = []
    for text in texts:
        inputs = tokenizer(
            [text],
            padding="max_length",
            max_length=tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            hidden = text_encoder(inputs.input_ids)[0] 
        # SCOUT: apply top-k sparse compression (simulates client upload) 
        if CFG.topk_k is not None:
            hidden = topk_compress_tensor(hidden, CFG.topk_k)
        per_caption_embs.append(hidden.detach().cpu().numpy())

    all_embs = np.concatenate(per_caption_embs, axis=0) 
    avg_emb_np = all_embs.mean(axis=0, keepdims=True)   

    embeddings_by_class[cls] = {"captions": texts, "embedding": avg_emb_np.tolist()}
    np.save(emb_dir / f"{cls}.npy", avg_emb_np)
    np.save(emb_dir / f"{cls}_all.npy", all_embs) 

    if (i + 1) % 5 == 0 or (i + 1) == len(selected_classes):
        print(f"  Embedded {i+1}/{len(selected_classes)} classes  ({len(texts)} captions each)")

emb_json_name = f"{CFG.dataset_name}_clip_embeddings_{topk_label}.json"
emb_path = WORK / "embeddings" / emb_json_name
with open(emb_path, "w") as f:
    json.dump(embeddings_by_class, f, indent=2)

out_emb_dir = OUTPUTS / "embeddings" / CFG.dataset_name
out_emb_dir.mkdir(parents=True, exist_ok=True)
for cls in selected_classes:
    for suffix in ["", "_all"]:
        src = emb_dir / f"{cls}{suffix}.npy"
        if src.exists():
            shutil.copy(src, out_emb_dir / src.name)
shutil.copy(emb_path, OUTPUTS / emb_json_name)

print(f"CLIP embeddings saved -> {emb_path}")
print(f"Per-class .npy files  -> {out_emb_dir}")
print("Avg embedding shape:", avg_emb_np.shape, "  Per-caption stack shape:", all_embs.shape)

del text_encoder, tokenizer
torch.cuda.empty_cache()


Loading SD-compatible CLIP model (768d)...


tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Generating CLIP embeddings for 12 classes (compression: topk64)...
  Embedded 5/12 classes  (20 captions each)
  Embedded 10/12 classes  (20 captions each)
  Embedded 12/12 classes  (20 captions each)
CLIP embeddings saved -> /kaggle/working/oscar_baseline_domainnet/embeddings/domainnet_clip_embeddings_topk64.json
Per-class .npy files  -> /kaggle/working/outputs/embeddings/domainnet
Avg embedding shape: (1, 77, 768)   Per-caption stack shape: (20, 77, 768)


## 7) Stable Diffusion synthesis

Generates synthetic images for every class using the CLIP embeddings computed above. Training will done in Part b notebook only on synthetic dataset for client privacy.

Embeddings are loaded from the `.npy` files saved in the previous step.

In [ ]:
import threading
from diffusers import StableDiffusionPipeline

def generate_on_gpu(gpu_id, gpu_device, client_classes_subset, emb_dir, save_dir, cfg, seed):
    print(f"[GPU {gpu_id}] Loading SD pipeline...")
    pipe = StableDiffusionPipeline.from_pretrained(
        cfg.sd_model,
        use_safetensors=True,
        safety_checker=None,
        requires_safety_checker=False,
        token=secret_value_0,
    )
    pipe = pipe.to(gpu_device)
    pipe.enable_attention_slicing()
    pipe.set_progress_bar_config(disable=True)
    print(f"[GPU {gpu_id}] Pipeline ready on {gpu_device}.")

    for cls in client_classes_subset:
        cls_dir = Path(save_dir) / cls
        cls_dir.mkdir(parents=True, exist_ok=True)
        blank_ids = torch.zeros(1, 77, dtype=torch.long, device=gpu_device)
        with torch.no_grad():
            uncond_emb = pipe.text_encoder(blank_ids)[0].to(torch.float16)
        
        all_npy = Path(emb_dir) / f"{cls}_all.npy"
        if all_npy.exists():
            all_embs = np.load(str(all_npy))
        else:
            all_embs = np.load(str(Path(emb_dir) / f"{cls}.npy"))
            print(f"[GPU {gpu_id}][warn] {cls}: _all.npy missing, using mean emb")
        n_embs = all_embs.shape[0]
        
        for i in range(cfg.synth_images_per_class):
            cap_emb = torch.from_numpy(
                all_embs[i % n_embs : i % n_embs + 1]
            ).to(gpu_device, dtype=torch.float16)
            gen = torch.Generator(device=gpu_device).manual_seed(seed + 1000 + i)
            with torch.autocast("cuda"):
                image = pipe(
                    prompt_embeds=cap_emb,
                    negative_prompt_embeds=uncond_emb,
                    num_inference_steps=cfg.diffusion_steps,
                    guidance_scale=cfg.guidance_scale,
                    generator=gen,
                ).images[0]
            image.save(cls_dir / f"synth_{i:03d}.jpg")

    del pipe
    torch.cuda.empty_cache()
    print(f"[GPU {gpu_id}] Generation complete.")


emb_dir_load = WORK / "embeddings" / CFG.dataset_name
if NUM_GPUS >= 2:
    # Two real GPUs: split classes, run in parallel threads
    mid = len(selected_classes) // 2
    classes_gpu0 = selected_classes[:mid]
    classes_gpu1 = selected_classes[mid:]
    print(f"Starting SD generation on 2 GPUs ({len(classes_gpu0)} + {len(classes_gpu1)} classes)...")
    t0 = threading.Thread(
        target=generate_on_gpu,
        args=(0, GPU0, classes_gpu0, emb_dir_load, WORK / "synthetic", CFG, SEED),
        name="sd-gpu0",
    )
    t1 = threading.Thread(
        target=generate_on_gpu,
        args=(1, GPU1, classes_gpu1, emb_dir_load, WORK / "synthetic", CFG,SEED),
        name="sd-gpu1",
    )
    t0.start(); t1.start()
    t0.join(); t1.join()
else:
    # Single GPU (Kaggle free T4): all classes sequentially on GPU0
    print(f"Starting SD generation on 1 GPU (sequential, {len(selected_classes)} classes)...")
    generate_on_gpu(0, GPU0, selected_classes, emb_dir_load, WORK / "synthetic", CFG, SEED)

print("All synthetic generation finished.")


Starting SD generation on 2 GPUs (6 + 6 classes)...
[GPU 0] Loading SD pipeline...
[GPU 1] Loading SD pipeline...


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

text_encoder/model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

[GPU 0] Pipeline ready on cuda:0.
[GPU 1] Pipeline ready on cuda:1.
[GPU 0] Generation complete.
[GPU 1] Generation complete.
All synthetic generation finished.


Synthetic dataset generated is now saved to notebook outputs. PartB notebook will be used to train and evaluate the results and accuracy of model. This allows reusability and parallelization of tasks on GPUs. 
